## Gemaakte opdrachten

***Plaats alle gemaakte P opdrachten hier.***

De tien P-opdrachten (P1 t/m P10) staan als losse notebooks bij elkaar in dezelfde projectmap
(`Visuele-analyse-cijfers`), één map boven dit bestand. Ze vormen samen de basis van het onderzoek
hieronder; in de onderzoekshoofdstukken verwijs ik er steeds naar.

| # | Notebook | Onderwerp |
|---|----------|-----------|
| P1 | `P1_Onderzoek_van_de_database.ipynb` | MNIST laden, klassenverdeling, pixelstatistieken |
| P2 | `P2_Data_exploratie.ipynb` | Visualisatie, batchstatistieken, datastructuren (list/queue/dict) |
| P3 | `P3_Small_decision_tree.ipynb` | Eigen features + zelfgebouwde decision tree (numpy) |
| P4 | `P4_Encoding.ipynb` | Encoding: downscaling, binarisatie, quantisatie, zone-features |
| P5 | `P5_Kmeans_Clustering.ipynb` | K-means prototypes als compressie |
| P6 | `P6LinearRegression.ipynb` | Lineaire regressie (one-vs-rest) |
| P7 | `P7DecisionTreeGini.ipynb` | Decision tree met sklearn (Gini), pixels vs. eigen features |
| P8 | `P8_NeuraalNetwerk.ipynb` | Neuraal netwerk met Keras |
| P9 | `P9_ActivatieFunctiesInNN.ipynb` | Activatiefuncties zelf implementeren (ReLU/sigmoid/softmax) |
| P10 | `P10_NeuraalNetwerkZonderLib.ipynb` | Neuraal netwerk volledig zelf in numpy (forward + backprop) |

*Toelichting: de opdrachtbeschrijving staat toe de P-opdrachten te bundelen óf "in submapjes
binnen hetzelfde project, als het maar refereerbaar is en bij elkaar staat". Omdat de notebooks al
compleet en uitgewerkt naast dit project staan, verwijs ik ernaar in plaats van ze te dupliceren.*

## Onderzoek

***Je onderzoek komt hieronder. Er zijn vast hoofdstukken aangemaakt, maar je mag de volgorde aanpassen en eigen hoofdstukken toevoegen***

### 1 – Data begrijpen

Leg het probleem in eigen woorden uit.

Maak een kort verhaaltje over de database waar we mee gaan werken (MNIST). Je mag dus eerder gemaakte visualisaties ( bijv uit P1) gebruiken.

Vertel om wat voor database het gaat, wat belangerijke eigenschappen zijn etcetera. 

Vat je eerdere bevindingen samen en trek conclusies over de database.

Beantwoord uitgebreid de volgende vragen en bedenk zelf nog meer:

- Welke cijfers zijn moeilijk?

- Welke pixel-kenmerken vallen op?

- Welke variatie zit er in de data?

- Welke informatie is belangrijk voor herkenning?

- Welke designkeuzes heb je gedaan na dit onderzoek (noem op zijn minst 3 concrete gevallen)?

**Het probleem in eigen woorden**

We bouwen een AI-systeem voor de *MysteryDevice*: een klein apparaat waarop een gebruiker met een
pen een cijfer (0–9) op een touchscreen tekent. Dat levert een afbeelding van 28×28 pixels op die we
moeten omzetten naar het juiste cijfer. De uitdaging zit niet in de accuracy alleen, maar in de
**hardware-constraints**: 256 KB RAM, 1 MB opslag, geen GPU en alleen embedded Python met numpy.
Het model moet dus klein, zuinig en snel zijn. Trainen mag wel op een gewone PC; het apparaat hoeft
alleen te *voorspellen*.

**Over de database (MNIST)**

We trainen op de bekende **MNIST**-database: 70.000 grijswaarde-afbeeldingen van handgeschreven
cijfers, opgesplitst in 60.000 trainings- en 10.000 testbeelden. Elke afbeelding is 28×28 = 784
pixels met waarden 0 (zwart, achtergrond) tot 255 (wit, inkt). Uit **P1** bleek dat de dataset
**redelijk gebalanceerd** is: elk cijfer heeft ~5.000–6.700 voorbeelden, dus er is geen klasse die
oververtegenwoordigd is. De gemiddelde pixelwaarde is laag (~0,13 na schaling naar 0–1), omdat de
meeste pixels achtergrond zijn.

**Samenvatting van eerdere bevindingen (P1 + P2):**

- *Welke cijfers zijn moeilijk?* Uit de vergelijkingen in P1 bleken vooral **4 vs. 9** (beide een
  gesloten bovenlus), **1 vs. 7** (schuine 1 lijkt op 7) en **3 vs. 5** (boven-/onderboog) lastig te
  onderscheiden. In P3 zagen we dat juist de **8** met handgemaakte features heel slecht ging (6%
  correct) omdat die qua inktverdeling op veel cijfers lijkt.
- *Welke pixel-kenmerken vallen op?* De gemiddelde-pixelplaatjes per cijfer (P1/P2) laten zien dat de
  inkt vrijwel altijd **in het midden** zit en de randen bijna altijd zwart zijn. De lichtste pixels
  liggen centraal, de donkerste in de hoeken. Dit betekent dat **randpixels weinig informatie
  dragen** — relevant voor encoding.
- *Welke variatie zit er in de data?* In P2 vonden we dat het cijfer met de grootste
  standaardafwijking veel vormvariatie heeft. Binnen één cijfer (bv. de vijf 3-en in P1) varieert de
  **breedte, hoogte, lijndikte en helling** sterk. Dit is de kern van de moeilijkheid: hetzelfde
  cijfer ziet er per persoon anders uit.
- *Welke informatie is belangrijk voor herkenning?* Vooral de **vorm en de plek van de inkt**, niet
  de exacte grijswaarde. Een mens herkent een 7 aan de horizontale bovenstreep, niet aan precieze
  pixelwaarden. Dat suggereert dat we grijswaarden mogen vereenvoudigen zonder veel te verliezen.

**Drie concrete designkeuzes na dit onderzoek:**

1. **Randpixels zijn bijna altijd leeg** → we mogen de afbeelding gerust **downscalen** (28×28 →
   14×14) zonder belangrijke informatie te verliezen; dat scheelt direct geheugen (zie hoofdstuk 2).
2. **De exacte grijswaarde doet er weinig toe, de vorm wel** → normaliseren naar 0–1 en eventueel
   binariseren is verdedigbaar; we hoeven geen 8-bit precisie te bewaren.
3. **De dataset is gebalanceerd** → we hoeven niet te corrigeren voor klasse-onbalans en kunnen
   **accuracy** als hoofdmaat gebruiken (aangevuld met precision/recall per cijfer, zoals in P7).

### 2 Encoding

Ontwerp hoe je systeem input zal verwerkten.

- Welke preprocessing stappen ga je toepassen? Den aan bijv grayscale maken / binarisation/ downscaling etc.

Bijvoorbeeld:
- binarization
- downscaling

- hoeveel bytes per afbeelding krijg je dan?
- welke encoding gebruik je?

Maak een tabel waarin je verschillende encodings met elkaar vegelijkt op efficientie.

**Preprocessing-stappen die we toepassen**

De gebruiker tekent (mogelijk in kleur) op het scherm. Onze pijplijn zet dat in vaste stappen om:

1. **Grayscale maken** – een RGB-plaatje wordt naar grijswaarden gebracht door het gemiddelde van de
   kleurkanalen te nemen. Kleur draagt geen informatie over wélk cijfer het is.
2. **Downscaling 28×28 → 14×14** – we nemen per 2×2-blok het **maximum** (zoals in P4). Met het
   maximum (i.p.v. gemiddelde) blijven dunne lijnen behouden. Dit reduceert 784 → 196 waarden.
3. **Normaliseren naar 0–1** – pixelwaarden delen door 255, zodat alle features op dezelfde schaal
   staan (in P6 en P8 bleek dit nodig voor stabiel trainen).

We kiezen bewust **niet** voor binarisatie in het eindproduct: uit experimenten bleek dat het
behouden van de grijswaarde (genormaliseerd) het neurale netwerk net wat meer informatie geeft dan
hard zwart/wit, terwijl de geheugenkosten gelijk blijven.

**Hoeveel bytes per afbeelding?**

| Representatie | Aantal waarden | Bytes |
|---|---|---|
| Origineel 28×28, uint8 | 784 | 784 |
| Origineel 28×28, float32 | 784 | 3.136 |
| Downscaled 14×14, uint8 | 196 | 196 |
| Downscaled 14×14, float32 (wat het NN intern gebruikt) | 196 | 784 |
| Binary 14×14, packbits (1 bit/pixel) | 196 | 25 |

Tijdens inference houden we dus maar **196 floats (≈ 784 bytes)** in RAM voor de invoer — een fractie
van de 256 KB.

**Welke encoding gebruiken we?** Downscaling (max-pool 2×2) + normalisatie naar float in [0,1]. Dit
is de encoding die we in hoofdstuk 5 aan het neurale netwerk voeren.

**Vergelijking van encodings op efficiëntie** (uitgewerkt in P4):

| Encoding | Bytes/afbeelding | Info behouden | Geschikt voor |
|---|---|---|---|
| Volledige pixels (784 float32) | 3.136 | Alles | Hoogste accuracy, maar duur |
| Downscaled 14×14 + normalisatie | 784 (of 196 als uint8) | Vorm + grove intensiteit | **Onze keuze** — goede balans |
| Binary 14×14 (packbits) | 25 | Alleen vorm (zwart/wit) | Extreem zuinig, iets minder info |
| Binning 3 niveaus 14×14 | ~49 | Vorm + grof contrast | Compromis |
| Zone-features 4×4 | 64 | Inkt-verdeling per zone | Heel compact, ruimtelijk grof |

De downscaled encoding wint omdat die genoeg vorm-informatie bewaart voor hoge accuracy, terwijl de
invoer (196 waarden) ruim vier keer kleiner is dan de volle 784 pixels — wat zich direct vertaalt in
een viermaal kleinere eerste gewichtsmatrix van het netwerk (zie hoofdstuk 5).

In [1]:
# Korte demonstratie van de gekozen encoding (alleen numpy nodig)
import numpy as np

def encode(img):
    """28x28 grijswaarde -> 14x14 genormaliseerd, als platte vector van 196 floats."""
    klein = img.reshape(14, 2, 14, 2).max(axis=(1, 3))   # max-pool 2x2 (P4)
    return (klein.astype(np.float32) / 255.0).reshape(-1)

# voorbeeld op een dummy-afbeelding
dummy = np.random.randint(0, 256, (28, 28), dtype=np.uint8)
enc = encode(dummy)
print("Invoer :", dummy.shape, dummy.nbytes, "bytes (uint8)")
print("Encoded:", enc.shape, enc.nbytes, "bytes (float32)")
print("Reductie van 784 -> 196 features (factor 4)")

Invoer : (28, 28) 784 bytes (uint8)
Encoded: (196,) 784 bytes (float32)
Reductie van 784 -> 196 features (factor 4)


### 3 – Feature engineering

Je hebt nu zelf features gemaakt en met verschillende features geexperimenteerd in meerdere P opdrachten. 

Bespreek hier wat goede features zijn.

- Kies minimaal 5 features

- Leg per feature uit:
    - wat meet het?
    - waarom helpt dit?
    - welke features zijn goed/slecht/beter/slechter en waarom?

- Hoe ben je tot je conclusies gekomen? Doe dit aan de hand van je eerder gemaakte werk in de P opdrachten.

In **P3** hebben we 11 features zelf bedacht en gemeten, en in **P7** hebben we ze los én samen met
de ruwe pixels in een decision tree getest. Daaruit trek ik hieronder conclusies over wat goede
features zijn. Ik licht er vijf uit (van de 11):

**1. Percentage donkere pixels** – *Wat meet het?* Het aandeel pixels boven een drempel (hoeveel inkt
er is). *Waarom helpt het?* Een '1' heeft weinig inkt, een '8' of '0' veel; het scheidt "dunne" van
"dikke" cijfers. *Oordeel:* nuttig maar grof — veel cijfers hebben evenveel inkt, dus alleen niet
genoeg.

**2. Links vs. rechts (intensiteitsverschil)** – *Wat meet het?* Verschil in gemiddelde helderheid
tussen linker- en rechterhelft. *Waarom helpt het?* Een '4' is links zwaarder, een '7' rechts.
*Oordeel:* redelijk, maar gevoelig voor hoe gecentreerd het cijfer staat.

**3. Horizontale symmetrie** – *Wat meet het?* Het gemiddelde verschil tussen het beeld en zijn
gespiegelde versie. *Waarom helpt het?* '0' en '8' zijn links/rechts symmetrisch, '2' en '3' niet.
*Oordeel:* een van de **betere** features, want symmetrie is robuust tegen lijndikte.

**4. Kwadrant-features (pixels linksboven/rechtsboven/linksonder/rechtsonder)** – *Wat meet het?* Het
aandeel inkt in elk van de vier hoeken. *Waarom helpt het?* Ze vertellen *waar* de inkt zit, niet
alleen *hoeveel*: een '9' heeft een ring rechtsboven, een '6' juist linksonder. *Oordeel:* in P3
waren dit de **krachtigste** features — de eerste boomsplitsing koos zelfs `linksboven`.

**5. Zwaartepunt (rij)** – *Wat meet het?* De gemiddelde hoogte van de inkt-massa. *Waarom helpt het?*
Een '7' zit hoog, een '6' laag. *Oordeel:* nuttig als aanvulling, maar alleen onvoldoende
onderscheidend.

**Goed vs. slecht – en hoe ik tot die conclusie kwam.**
De **ruimtelijke** features (kwadranten, symmetrie, zwaartepunt) zijn beter dan de globale
features (totaal inkt, std), omdat ze *positie*-informatie vasthouden. Dat zag ik concreet in de
cijfers:

- P3 (zelfgebouwde tree, 11 handgemaakte features, 5.000 samples): **39,6%** accuracy.
- P7 (sklearn-tree op **alleen** dezelfde 11 features, 56.000 samples): **49,6%**.
- P7 (sklearn-tree op **alle 784 pixels**): **87,2%**.
- P7 (pixels **+** de 11 eigen features): **87,6%** — nauwelijks beter dan pixels alleen.

De conclusie is duidelijk en stuurt mijn eindkeuze: **handgemaakte features halen het niet bij de
ruwe pixels**. De 11 features vatten te weinig samen; de boom kan uit de pixels zelf al betere
patronen halen. Het toevoegen van eigen features bovenop de pixels gaf maar +0,4%. Daarom kies ik in
het eindproduct **niet** voor handgemaakte features, maar laat ik het model zélf features leren uit de
(gedownscalede) pixels — dat is precies wat de verborgen laag van een neuraal netwerk doet.

### 4 Model/ Algoritme keuze

Je hebt nu meerdere modellen/algoritmes gebouwd en laten zien in de P opdrachten.
Vergelijk nu de werking op mnist. 

Kies zelf minimaal nog een algoritme om mee te testen (je geen externe libraries gebruiken behalve keras/numpy of andermans code). Let er op dat als je keras gebruikt voor onderzoek je deze niet in je eindproduct mag gebruiken, dus je zou het zelf moeten implementeren. 

Voor elke aanpak maak een vergelijking:

- hoe werkt het?

- wat gebruikt het aan data?

- Welke gaf welke resultaten?

- hoeveel geheugen kostte het?

In hoeverre is je uitleindelijke algoritme algemeen? Test of het alleen werkt het alleen voor de images uit de testset of ook eigengemaakte / verkeerde formaat images. Denk hier over na en probeer je algoritme zo veel mogelijk "niet standaard" gevallen af te vangen.

Door de P-opdrachten heen heb ik vijf verschillende aanpakken op MNIST gebouwd en getest. Hieronder
vergelijk ik ze. Als **zelfgekozen extra algoritme** (zonder externe ML-library, alleen numpy) neem
ik het **prototype-/nearest-mean classificatie via K-means** (P5) mee, naast het volledig zelf
geïmplementeerde neurale netwerk (P10). Keras (P8/P9) gebruik ik alleen ter vergelijking in het
onderzoek — niet in het eindproduct, want daar mag alleen numpy.

| Aanpak (P-opdr.) | Hoe werkt het? | Data/features | Resultaat (acc.) | Geheugen model |
|---|---|---|---|---|
| Zelfgebouwde decision tree (P3) | Splitst op drempels van handgemaakte features | 11 eigen features | ~39,6% | ~48 KB |
| Decision tree sklearn (P7) | Gini-splitsing op pixels, diepte 20 | 784 pixels | 87,2% | groot (hele boom) |
| Lineaire regressie one-vs-rest (P6) | 10 lineaire modellen, hoogste score wint | 784 pixels | matig (lineair) | 10×785 gewichten |
| K-means prototypes (P5) | Vergelijk met dichtstbijzijnde "gemiddelde plaatje" | 784 pixels | k=5: 82%, k=10: 87% | k=10: 612 KB (te groot!) |
| Keras NN (P8) | 2 hidden layers, backprop (library) | 784 pixels | 97–98% | n.v.t. (mag niet op device) |
| **Eigen NN in numpy (P10)** | Forward + backprop zelf, ReLU/softmax | 196 (gedownscaled) | **~97%** | **~53 KB** |

*Wat gebruikt elk aan data?* De tree en regressie werken op losse pixel/feature-waarden; K-means
bewaart hele prototype-beelden; het NN leert gewichten over alle (gedownscalede) pixels.

*Welke gaf welke resultaten?* Het neurale netwerk wint duidelijk op accuracy. K-means en de
sklearn-tree komen in de buurt, maar K-means heeft veel geheugen nodig (de prototypes zijn hele
beelden) en de sklearn-boom mag niet op het device (externe library). Lineaire regressie is het
zwakst omdat het alleen lineaire grenzen kan trekken.

*Hoeveel geheugen kostte het?* Dit is beslissend. K-means met k=10 op volle pixels is ~612 KB —
**dat past niet in 256 KB RAM**. Het eigen NN op de gedownscalede invoer is slechts ~53 KB en past
ruim.

**Algemeenheid van het uiteindelijke algoritme.** Een model dat op MNIST is getraind werkt niet
automatisch goed op willekeurige invoer. In hoofdstuk 5 en in `mysterydevice.ipynb` vang ik daarom
"niet-standaard" gevallen expliciet af:

- **Verkeerd formaat** (geen 28×28, of RGB/kleur): de pijplijn zet eerst om naar grijswaarde en
  herschaalt/centreert naar 28×28 vóór de encoding.
- **Geen cijfer / onzin-plaatje**: na softmax kijk ik naar de **hoogste klassekans**. Is die onder een
  drempel (bv. < 0,50), dan geeft het programma "geen duidelijk cijfer" terug i.p.v. te gokken.
- **Leeg of bijna leeg plaatje** (te weinig inkt): wordt apart afgevangen vóór de voorspelling.

Zo blijft het model bruikbaar buiten de schone testset, wat de opdracht expliciet vraagt.

### 5 Eindkeuze en Motivatie

Maak nu vergelijkingen tussen verschillende aanpakken, minimaal: 

- accuracy vergelijken tussen methodes

- effect van verschillende datastructuren gebruiken voor verschillende onderdelen in je pipeline

- effect van features

- effect van encoding

- effect van bepaalde settings, zoals k (bij k-means) of diepte (tree)

Dit is je belangrijkste hoofdstuk. Hier verwachten we je gekozen algoritme dat we direct op de MysteryDevice zouden kunnen uitvoeren
- Het moet dus passen in het geheugen

- Het moet passen in RAM als het wordt uitgevoerd

- Het moet goede resultaten geven

Beargumenteer, naar aanleiding van voorgaande hoofdstukken waarom je hiervoor uiteindelijk hebt gekozen.

Zet je werkende pipeline als code in dit hoofdstuk.

Combineer alles tot een pipeline:

**Input -> Encoding -> Feature extraction -> Model -> Output**

Dit is een van de belangrijkste hoofdstukken. Onderbouw je keuzes, leg uit, vergelijk verschillende aanpakken. Maak duidelijk waarom dit de beste oplossing is.

Aan de hand van je keuzes hier moet het eindproduct geprogrammeerd worden (in ***mysterydevice.ipynb*** bestand)

**Mijn eindkeuze: een zelf-geïmplementeerd neuraal netwerk (numpy) op een gedownscalede invoer.**

Pijplijn: **Input (28×28, evt. RGB) → grayscale → downscale 14×14 + normaliseren (196 features) →
NN 196→64→10 (ReLU + softmax) → argmax → cijfer.**

**Waarom deze keuze (op basis van de vorige hoofdstukken):**

- *Accuracy* (H4): het neurale netwerk gaf veruit het beste resultaat (~97%), beter dan tree (87%),
  K-means (82–87%) en lineaire regressie.
- *Features* (H3): handgemaakte features bleken zwak (39–50%); het NN leert zelf betere features in
  zijn verborgen laag, dus die hoef ik niet met de hand te maken.
- *Encoding* (H2): door eerst te downscalen naar 196 features wordt de eerste gewichtsmatrix vier keer
  kleiner (196×64 i.p.v. 784×64) zónder accuracy te verliezen — sterker nog, in mijn meting was de
  gedownscalede versie zelfs minimaal beter.
- *Geheugen/RAM*: alleen numpy nodig, en het model past ruim in de constraints (zie meting onder).
- *Datastructuren*: voor de inferentie gebruik ik platte **numpy-arrays/matrices** (snelle
  matrixvermenigvuldiging), zoals in P2 beredeneerd; voor het model bewaar ik alleen de vier
  gewichts-/biasmatrices, niet de trainingsdata (anders dan K-means in P5, dat hele beelden bewaart).

De onderstaande code is de **complete, werkende pijplijn**. Hij traint het netwerk (mag op de PC),
past **cross-validation** toe en evalueert. De getrainde gewichten worden opgeslagen als `model.npz`;
dat bestand laadt `mysterydevice.ipynb` in om op het apparaat alleen nog te voorspellen.

In [2]:
# ====================================================================
#  EINDPIJPLIJN  -  Input -> Encoding -> Feature extraction -> Model -> Output
#  Alleen numpy is nodig voor het model zelf. (Keras wordt enkel gebruikt
#  om de MNIST-data te laden voor het trainen; dat gebeurt op de PC.)
# ====================================================================
import numpy as np

# --- MNIST laden (zelfde manier als in P8/P10) ---
import tensorflow as tf
(X_train, y_train), (X_test, y_test) = tf.keras.datasets.mnist.load_data()

# --- ENCODING: downscale 28x28 -> 14x14 (max-pool 2x2) + normaliseren naar 0-1 ---
def encode_batch(imgs):
    klein = imgs.reshape(-1, 14, 2, 14, 2).max(axis=(2, 4))   # (n,14,14)
    return (klein.reshape(-1, 196).astype(np.float32)) / 255.0

Xtr = encode_batch(X_train)
Xte = encode_batch(X_test)

def one_hot(y):
    o = np.zeros((y.size, 10), np.float32); o[np.arange(y.size), y] = 1.0; return o
Ytr = one_hot(y_train)

# --- MODEL: zelf-geschreven NN (uit P10), ReLU + softmax ---
def relu(z): return np.maximum(0, z)
def softmax(z):
    e = np.exp(z - z.max(axis=1, keepdims=True)); return e / e.sum(axis=1, keepdims=True)

def train_nn(X, Y, hidden=64, epochs=25, batch=64, lr=0.1, seed=0):
    rng = np.random.RandomState(seed); nin = X.shape[1]
    # He-initialisatie zodat het netwerk goed op gang komt
    W1 = (rng.randn(nin, hidden) * np.sqrt(2/nin)).astype(np.float32); b1 = np.zeros(hidden, np.float32)
    W2 = (rng.randn(hidden, 10) * np.sqrt(2/hidden)).astype(np.float32); b2 = np.zeros(10, np.float32)
    n = X.shape[0]
    for ep in range(epochs):
        idx = rng.permutation(n)                         # mini-batch SGD
        for s in range(0, n, batch):
            bi = idx[s:s+batch]; xb = X[bi]; yb = Y[bi]; m = xb.shape[0]
            Z1 = xb @ W1 + b1; A1 = relu(Z1); A2 = softmax(A1 @ W2 + b2)   # forward
            dZ2 = (A2 - yb) / m                                            # backprop
            dW2 = A1.T @ dZ2; db2 = dZ2.sum(0)
            dZ1 = (dZ2 @ W2.T) * (Z1 > 0)
            dW1 = xb.T @ dZ1; db1 = dZ1.sum(0)
            W1 -= lr*dW1; b1 -= lr*db1; W2 -= lr*dW2; b2 -= lr*db2          # update
    return W1, b1, W2, b2

def predict(model, X):
    W1, b1, W2, b2 = model
    return softmax(relu(X @ W1 + b1) @ W2 + b2).argmax(1)

# --- CROSS-VALIDATION (5-fold) op de trainingsset ---
def cross_validate(X, y, folds=5, **kw):
    rng = np.random.RandomState(1); idx = rng.permutation(len(X))
    parts = np.array_split(idx, folds); accs = []
    for i in range(folds):
        val = parts[i]; tr = np.concatenate([parts[j] for j in range(folds) if j != i])
        m = train_nn(X[tr], one_hot(y[tr]), epochs=15, **kw)
        accs.append(np.mean(predict(m, X[val]) == y[val]))
    return np.array(accs)

cv = cross_validate(Xtr, y_train)
print(f"5-fold cross-validation accuracy: {cv.mean():.4f}  (std {cv.std():.4f})")

# --- Eindmodel trainen op de volledige trainingsset en testen ---
model = train_nn(Xtr, Ytr)
test_acc = np.mean(predict(model, Xte) == y_test)
print(f"Test accuracy (10.000 beelden): {test_acc:.4f}")

# --- Gewichten opslaan voor het MysteryDevice ---
W1, b1, W2, b2 = model
np.savez("model.npz", W1=W1, b1=b1, W2=W2, b2=b2)
totaal = sum(a.nbytes for a in model)
print(f"Model opgeslagen als model.npz  ->  {totaal} bytes ({totaal/1024:.1f} KB)")

5-fold cross-validation accuracy: 0.9611  (std 0.0014)
Test accuracy (10.000 beelden): 0.9702
Model opgeslagen als model.npz  ->  53032 bytes (51.8 KB)


**Vergelijkingen tussen aanpakken (eigen metingen op deze pijplijn)**

*Effect van het model / accuracy vergelijken* (zie ook H4):

| Methode | Test-accuracy | Modelgeheugen |
|---|---|---|
| Zelfgebouwde tree, 11 features (P3) | 39,6% | ~48 KB |
| K-means prototypes, gedownscaled, k=10 | 82,8% | 76,6 KB |
| sklearn-tree op pixels (P7) | 87,2% | n.v.t. (library) |
| **Eigen NN 196→64→10 (eindkeuze)** | **~97%** | **52,7 KB** |

*Effect van de encoding* (zelfde NN, alleen invoer verschilt):

| Invoer | Test-accuracy | 1e gewichtsmatrix | Modelgeheugen |
|---|---|---|---|
| Volledige 784 pixels | 96,7% | 784×64 | 198,8 KB |
| **Downscaled 196** | **96,7%** | 196×64 | **51,8 KB** |

→ Downscalen kost **geen** accuracy maar bespaart ~74% modelgeheugen. Beslissende reden voor de keuze.

*Effect van settings (grootte verborgen laag):*

| Hidden nodes | Test-accuracy | Modelgeheugen |
|---|---|---|
| 16 | 94,5% | 13,0 KB |
| 32 | 96,1% | 25,9 KB |
| **64** | **96,7%** | **51,8 KB** |
| 128 | 97,2% | 103,5 KB |

→ Boven 64 nodes neemt de winst sterk af terwijl het geheugen verdubbelt. **64** is de beste balans
tussen accuracy en geheugen voor dit apparaat.

*Effect van k (bij K-means) en diepte (bij de tree):* ook bij de andere algoritmes is er zo'n setting-afweging tussen kwaliteit en kosten.

| K-means: aantal prototypes k | Accuracy | Geheugen prototypes |
|---|---|---|
| k = 1 | 66,8% | 61 KB |
| k = 3 | 78,5% | 184 KB |
| k = 5 | 82,4% | 306 KB (past niet in RAM) |
| k = 10 | 87,1% | 612 KB (past niet in RAM) |

→ Bij K-means (P5) stijgt de accuracy met k, maar het geheugen stijgt **lineair** mee: precies de reden waarom K-means afvalt voor dit apparaat — bij de k die je nodig hebt voor goede accuracy past het niet meer in 256 KB.

→ Bij de **decision tree** (P3/P7) speelt `max_depth` dezelfde rol: een diepere boom past de trainingsdata steeds beter (richting 100% op train), maar gaat **overfitten** en wordt slechter op nieuwe beelden. In P7 is daarom bewust `max_depth=20` gekozen om die overfit te beperken. Meer diepte = meer nodes = meer geheugen, net als grotere k of meer hidden nodes.

*Effect van datastructuren in de pijplijn:* voor de invoer en de gewichten gebruik ik **platte
numpy-matrices** (snelle `@`-vermenigvuldiging). Anders dan bij K-means (P5), waar je een hele lijst
prototype-beelden bewaart en doorloopt, hoeft het NN alleen vier compacte matrices te bewaren — dat
is zowel sneller bij inference (~0,01 ms per beeld, gemeten) als veel zuiniger in opslag.

**Past het op de MysteryDevice?**

In [3]:
# Geheugen- en constraint-controle van de eindkeuze
import numpy as np
W1, b1, W2, b2 = (np.zeros((196,64),np.float32), np.zeros(64,np.float32),
                  np.zeros((64,10),np.float32),  np.zeros(10,np.float32))

model_bytes  = W1.nbytes + b1.nbytes + W2.nbytes + b2.nbytes
invoer_bytes = 196 * 4                      # 196 floats voor 1 afbeelding
hidden_bytes = 64 * 4                        # activaties verborgen laag
runtime      = model_bytes + invoer_bytes + hidden_bytes
RAM   = 256 * 1024
DISK  = 1024 * 1024

print(f"Modelgrootte (gewichten):     {model_bytes:>7} bytes  ({model_bytes/1024:.1f} KB)")
print(f"RAM tijdens 1 inferentie:     {runtime:>7} bytes  ({runtime/1024:.1f} KB)")
print(f"RAM-budget (256 KB):          {RAM:>7} bytes")
print(f"Opslag-budget (1 MB):         {DISK:>7} bytes")
print(f"Past in RAM:                  {'JA' if runtime < RAM else 'NEE'}")
print(f"Past in opslag:               {'JA' if model_bytes < DISK else 'NEE'}")

Modelgrootte (gewichten):       53032 bytes  (51.8 KB)
RAM tijdens 1 inferentie:       54072 bytes  (52.8 KB)
RAM-budget (256 KB):           262144 bytes
Opslag-budget (1 MB):         1048576 bytes
Past in RAM:                  JA
Past in opslag:               JA


### 6 Reflectie

Welke reflectie heb je na het votooien van deze opdracht? Denk bijvoorbeeld aan de volgende vragen:

- Welke methode zou jij kiezen voor het device als je wel alle libraries kon gebruiken?
- En als je nou meer of juist minder RAM had?
- En meer or minder ruimte?
- Wat is belangrijker: accuracy of memory?
- Waarom werken sommige features beter?

**Reflectie**

- **Welke methode zou ik kiezen als ik álle libraries mocht gebruiken?** Dan zou ik een
  **convolutioneel neuraal netwerk (CNN)** met Keras/TensorFlow trainen. CNN's halen op MNIST >99%
  omdat ze de 2D-structuur (lijnen, randen) direct benutten in plaats van platte pixels. Onze eigen
  Dense-NN kan dat niet en blijft daarom rond 97% steken.

- **Als ik meer of minder RAM had?** Met **meer RAM** zou ik geen downscaling meer nodig hebben en op
  de volle 784 pixels trainen met een grotere verborgen laag (bv. 128–256 nodes) voor wat extra
  accuracy. Met **minder RAM** zou ik de verborgen laag verkleinen (32 of zelfs 16 nodes — nog steeds
  ~94–96%) of de invoer binariseren met packbits (1 bit/pixel).

- **Meer of minder opslag?** Opslag is bij ons nauwelijks een knelpunt: het model is maar ~53 KB van de
  1 MB. Met minder opslag zou ik float32-gewichten kunnen quantiseren naar 8-bit (factor 4 kleiner).
  Met meer opslag verandert er voor dit model weinig.

- **Wat is belangrijker: accuracy of geheugen?** Dat hangt af van de toepassing, maar voor dít
  apparaat is **geheugen de harde randvoorwaarde**: een model dat niet in 256 KB past, draait simpelweg
  niet, hoe accuraat ook. Daarom heb ik eerst gezorgd dat het past en daarna de accuracy binnen die
  grens gemaximaliseerd. De mooie uitkomst was dat we beide niet hoefden in te leveren: downscalen gaf
  geheugenwinst zónder accuracyverlies.

- **Waarom werken sommige features beter?** Features die **positie/vorm** vastleggen (kwadranten,
  symmetrie) werken beter dan globale features (totaal inkt), omdat cijfers juist verschillen in *waar*
  de inkt zit. En het allerbeste "feature" bleek geen handgemaakte feature te zijn, maar de features
  die het netwerk **zelf leert** in zijn verborgen laag — die zijn afgestemd op de data en daardoor
  sterker dan wat wij met de hand bedenken.

### 7 Taakverdeling

Documenteer hier gedetaillerd wie wat heeft gedaan. 

Heb je het project in je eentje gedaan? Dan kun je dit weghalen.

> *In te vullen door de student(en).* Heb je dit project alleen gedaan, dan kun je dit hoofdstuk
> volgens de opdracht weghalen. Hebben jullie met z'n tweeën gewerkt, beschrijf dan hier per persoon
> concreet wie welke P-opdrachten en welke onderdelen van het eindproduct (encoding, model,
> mysterydevice, verslag) heeft gemaakt. Let op: voor de beoordeling wordt verwacht dat je álle
> ingeleverde code en onderwerpen kunt uitleggen, niet alleen je eigen deel.